# LSTM predictions vs realized

- **One table:** all tickers — `pred_*` vs `y_*` as % (— if missing).
- **Charts:** one figure per ticker — x-axis **3d, 7d, 30d, 90d, 180d**; **solid line + circles = realized**, **dashed line + squares = predicted**.
- **Price/vol:** last ~6 months of normalized close vs 20d ann. vol.

Uses `your_data.pkl` (via `load_panel_long`) and trained models in `src_2/saved_models/`.

In [1]:
%matplotlib inline
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

# Run from _my_builder_2/:  python auto.py  (see builders/_my_builder_2/README.md)


def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "src_2" / "_u_entries.py").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError("Run from repo root or src_2/notebooks/.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)


ROOT = /Users/yerik/_apple_lib/_peg_ProgEnvGit/a4yy_STRATEGY


In [2]:
MONTHS_HISTORY = 6
TICKERS: list[str] | None = None
MAX_PLOT: int | None = None

from src_2 import _u_entries as U

HORIZON_KEYS = list(U.HORIZON_DAYS.keys())
HORIZON_LABELS = [str(k) for k in HORIZON_KEYS]

In [3]:
from src_2._0_fns_io import load_panel_long
from src_2._1_fns_targets import add_return_targets
from src_2._2_fns_features import prepare_ml_frame, tickers_eligible_for_ml
from src_2._5_fns_predict import collect_predictions_table

df = load_panel_long()
df = add_return_targets(df)
df["date"] = pd.to_datetime(df["date"])
mx = df["date"].max()
d0 = mx - pd.DateOffset(months=MONTHS_HISTORY)
win = df[(df["date"] >= d0) & (df["date"] <= mx)].copy()
print("Panel:", df["date"].min(), "→", mx)

Panel: 2021-04-19 00:00:00 → 2026-04-18 00:00:00


In [4]:
if TICKERS is None:
    tickers = tickers_eligible_for_ml()
else:
    tickers = [str(t).upper().strip() for t in TICKERS]
if MAX_PLOT is not None:
    tickers = tickers[: int(MAX_PLOT)]
print(len(tickers), "tickers")

21 tickers


In [5]:
%%time
df_scaled, _, _ = prepare_ml_frame(ticker_allowlist=tickers)
pred_df, skip_notes = collect_predictions_table(df_scaled, tickers, HORIZON_KEYS)
if skip_notes:
    print("Stale/skip:", len(skip_notes))

CPU times: user 16.7 s, sys: 119 ms, total: 16.9 s
Wall time: 17 s


In [6]:
last = df_scaled["date"].max()
truth = df_scaled.loc[df_scaled["date"] == last, ["date", "ticker"] + U.TARGET_COLS]
snap = truth.merge(pred_df, on=["date", "ticker"], how="inner")
snap = snap.sort_values("ticker").reset_index(drop=True)


def _fmt_pct(v) -> str:
    if v is None:
        return "—"
    try:
        if pd.isna(v):
            return "—"
    except Exception:
        pass
    return f"{100.0 * float(v):+.2f}%"


rows = []
for _, r in snap.iterrows():
    row = {"asof_date": str(r["date"])[:10], "ticker": r["ticker"]}
    for k in HORIZON_KEYS:
        row[f"pred_{k}"] = _fmt_pct(r.get(f"pred_{k}"))
        row[f"y_{k}"] = _fmt_pct(r.get(f"y_{k}"))
    rows.append(row)

table_full = pd.DataFrame(rows)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
try:
    from IPython.display import display

    display(table_full)
except ImportError:
    print(table_full.to_string())

,asof_date,ticker,pred_3d,y_3d,pred_7d,y_7d,pred_30d,y_30d,pred_90d,y_90d,pred_180d,y_180d
0,2025-10-20,AAPL,+0.92%,-1.01%,+0.55%,+2.51%,-0.44%,+2.41%,-0.72%,-2.56%,-2.41%,+3.05%
1,2025-10-20,ACHR,+0.43%,-8.93%,-0.93%,-5.34%,-0.23%,-37.65%,+1.15%,-26.04%,-8.24%,-49.00%
2,2025-10-20,AMD,-4.58%,-2.32%,+1.00%,+7.94%,+2.67%,-7.07%,+0.69%,-3.63%,-3.11%,+15.73%
3,2025-10-20,ANDE,-1.56%,-1.96%,-0.34%,-1.49%,-0.50%,+4.04%,+7.13%,+21.76%,+19.27%,+51.78%
4,2025-10-20,ASML,+0.23%,-0.55%,-0.71%,+1.71%,+0.52%,-0.27%,-1.23%,+30.36%,-10.27%,+40.08%
5,2025-10-20,AVGO,-1.75%,-1.42%,+0.92%,+3.67%,+3.43%,+1.48%,+20.02%,+0.71%,+26.96%,+16.41%
6,2025-10-20,COIN,-2.08%,-6.11%,-2.14%,+5.13%,-1.13%,-25.16%,-2.52%,-29.85%,+3.39%,-39.98%
7,2025-10-20,CRSP,-1.70%,-9.14%,-3.74%,-11.15%,-5.39%,-31.42%,-6.10%,-27.66%,+8.97%,-21.60%
8,2025-10-20,HOOD,+1.60%,-1.08%,-0.56%,+7.41%,+9.81%,-12.99%,+12.57%,-19.93%,+36.32%,-33.17%
9,2025-10-20,NFLX,-0.26%,-10.09%,-0.13%,-11.63%,-0.71%,-11.19%,+4.49%,-28.95%,+1.98%,-21.43%


In [ ]:
# Pred vs real charts: removed while testing (no src_2._7 module). Use menu "2 — Predict" or ad-hoc plots in a new cell.
pass


In [ ]:
def plot_price_vol(ticker: str) -> None:
    g = win[win["ticker"] == ticker].sort_values("date")
    if g.empty:
        return
    ret = g["close"].pct_change()
    ann_vol = ret.rolling(20, min_periods=5).std() * np.sqrt(252.0)
    px = g["close"] / g["close"].iloc[0]
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(g["date"], px, color="C0", label="Close (norm)")
    ax.set_ylabel("Norm close", color="C0")
    ax.tick_params(axis="y", labelcolor="C0")
    ax2 = ax.twinx()
    ax2.plot(g["date"], ann_vol, color="C1", alpha=0.85)
    ax2.set_ylabel("Ann. vol 20d", color="C1")
    ax2.tick_params(axis="y", labelcolor="C1")
    ax.set_title(f"{ticker} — ~{MONTHS_HISTORY}m")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()


for sym in tickers:
    plot_price_vol(sym)